# How to Fine-Tune LLMs with LoRA Adapters using Hugging Face TRL

This notebook demonstrates how to efficiently fine-tune large language models using LoRA (Low-Rank Adaptation) adapters. LoRA is a parameter-efficient fine-tuning technique that:
- Freezes the pre-trained model weights
- Adds small trainable rank decomposition matrices to attention layers
- Typically reduces trainable parameters by ~90%
- Maintains model performance while being memory efficient

We'll cover:
1. Setup development environment and LoRA configuration
2. Create and prepare the dataset for adapter training
3. Fine-tune using `trl` and `SFTTrainer` with LoRA adapters
4. Test the model and merge adapters (optional)


## 1. Setup development environment

Our first step is to install Hugging Face Libraries and Pytorch, including trl, transformers and datasets. If you haven't heard of trl yet, don't worry. It is a new library on top of transformers and datasets, which makes it easier to fine-tune, rlhf, align open LLMs.


In [13]:
# Install the requirements in Google Colab
!pip install transformers datasets trl huggingface_hub peft

# Authenticate to Hugging Face
import os
from huggingface_hub import login
#from dotenv import load_dotenv
from google.colab import userdata

# Authenticate to Hugging Face
#load_dotenv()
login(token=userdata.get('HF_TOKEN'))

## 2. Load the dataset
The dataset can be found under this URL `https://www.kaggle.com/datasets/pes12017000148/food-ingredients-and-recipe-dataset-with-images/data`

In [14]:
# Load a sample dataset
from datasets import load_dataset

# TODO: define your dataset and config using the path and name parameters
dataset_smol = load_dataset(path="HuggingFaceTB/smoltalk", name="everyday-conversations")
print(dataset_smol)
cols_to_drop = ["Unnamed: 0"]
dataset_ = load_dataset(
    "csv",
    data_files={
        "train": "13k-recipes.csv",
    },
).remove_columns(cols_to_drop).shuffle(seed=42)["train"]#.take(3000)
print(dataset_)

DatasetDict({
    train: Dataset({
        features: ['full_topic', 'messages'],
        num_rows: 2260
    })
    test: Dataset({
        features: ['full_topic', 'messages'],
        num_rows: 119
    })
})
Dataset({
    features: ['Title', 'Ingredients', 'Instructions', 'Image_Name', 'Cleaned_Ingredients'],
    num_rows: 13501
})


In [15]:
for element in dataset_.take(3).to_pandas()[["Title", "Instructions", "Cleaned_Ingredients"]].values:
    # The dataset is not perfect, but it is a good starting point.
    # Results could likely be improved by removing measurements or finding the serving size.
    print(10*"#")
    print(element[0])
    print(10*"-")
    print(element[1])
    print(10*"-")
    print(element[2])

##########
Gin Rocket
----------
Using a mandoline or vegetable peeler, thinly slice enough of the fennel bulb to yield 1/4 cup (about 1/2 small bulb). Add a pinch of the fennel fronds to a cocktail shaker along with sliced fennel and arugula leaves. Add lime juice and simple syrup and muddle until the fennel is bruised. Add gin and fill with ice. Shake vigorously until chilled, about 12 seconds. Double-strain into a chilled coupe glass and garnish with lime wheel, arugula leaf, or nasturtium if desired.
----------
['1 fennel bulb with fronds', '1/4 cup packed arugula leaves', '3/4 ounce fresh lime juice', '3/4 ounce 1:1 simple syrup (see note)', '2 ounces gin', 'GARNISH: lime wheel', 'arugula leaf', 'or peppery flower (such as nasturtium; optional)']
##########
Rosemary Pork Chops
----------
Preheat broiler.
Mince and mash garlic to a paste with a pinch of salt, then stir together with rosemary, oil, 3/4 teaspoon salt, and 1/2 teaspoon pepper. Rub mixture all over chops.
Broil chops o

In [16]:
system_message = """You are an AI assistant tasked with extracting ingredients from a recipe text. Here's what you need to do:

1. Read the following recipe text carefully.

2. Identify all ingredients mentioned in the text. Look for:
   - Specific food items
   - Quantities (e.g., numbers, fractions, or words like "cup", "tablespoon", "pinch")
   - Any descriptors of the ingredients (e.g., "chopped", "diced", "fresh")

3. Create a list of all identified ingredients. Each ingredient should be a complete phrase including its quantity and any descriptors.

4. Output your list of ingredients as a Python list of strings. Each ingredient should be a separate element in the list.

For example:
['3 shallots', '6 tablespoons (3/4 stick) butter, room temperature', '4 teaspoons minced fresh rosemary', '1 tablespoon grated orange peel']

Remember to include all ingredients, even if they seem minor or are used in small quantities.
"""

def process_dataset(sample):
    # Convert the sample into a chat format
    sample = {
        "messages": [
            {"role": "system", "content": system_message},
            {"role": "user", "content": f'Extract all ingredients from this recipes: {sample["question"]}'},
            {"role": "assistant", "content": sample["answer"]}
        ]
    }
    return sample

In [17]:
dataset = dataset_.rename_column("Instructions", "question")
dataset = dataset.rename_column("Cleaned_Ingredients", "answer")
dataset = dataset.map(process_dataset, remove_columns=dataset.features)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

# add timestamp to filename
import datetime
timestamp = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
dataset["train"].to_json(f"sft_data/train_dataset_{timestamp}.json", orient="records")
dataset["test"].to_json(f"sft_data/test_dataset_{timestamp}.json", orient="records")
dataset["train"][2]["messages"]

Creating json from Arrow format:   0%|          | 0/13 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

[{'content': 'You are an AI assistant tasked with extracting ingredients from a recipe text. Here\'s what you need to do:\n\n1. Read the following recipe text carefully.\n\n2. Identify all ingredients mentioned in the text. Look for:\n   - Specific food items\n   - Quantities (e.g., numbers, fractions, or words like "cup", "tablespoon", "pinch")\n   - Any descriptors of the ingredients (e.g., "chopped", "diced", "fresh")\n\n3. Create a list of all identified ingredients. Each ingredient should be a complete phrase including its quantity and any descriptors.\n\n4. Output your list of ingredients as a Python list of strings. Each ingredient should be a separate element in the list.\n\nFor example:\n[\'3 shallots\', \'6 tablespoons (3/4 stick) butter, room temperature\', \'4 teaspoons minced fresh rosemary\', \'1 tablespoon grated orange peel\']\n\nRemember to include all ingredients, even if they seem minor or are used in small quantities.\n',
  'role': 'system'},
 {'content': 'Extract a

In [18]:
from datasets import load_dataset
dataset = load_dataset(
    "json",
    data_files={
        "train": "sft_data/train_*.json",
        "test": "sft_data/test_*.json",
    },
)
dataset

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 12150
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 1351
    })
})

## 3. Fine-tune LLM using `trl` and the `SFTTrainer` with LoRA

The [SFTTrainer](https://huggingface.co/docs/trl/sft_trainer) from `trl` provides integration with LoRA adapters through the [PEFT](https://huggingface.co/docs/peft/en/index) library. Key advantages of this setup include:

1. **Memory Efficiency**:
   - Only adapter parameters are stored in GPU memory
   - Base model weights remain frozen and can be loaded in lower precision
   - Enables fine-tuning of large models on consumer GPUs

2. **Training Features**:
   - Native PEFT/LoRA integration with minimal setup
   - Support for QLoRA (Quantized LoRA) for even better memory efficiency

3. **Adapter Management**:
   - Adapter weight saving during checkpoints
   - Features to merge adapters back into base model

We'll use LoRA in our example, which combines LoRA with 4-bit quantization to further reduce memory usage without sacrificing performance. The setup requires just a few configuration steps:
1. Define the LoRA configuration (rank, alpha, dropout)
2. Create the SFTTrainer with PEFT config
3. Train and save the adapter weights


In [19]:
# Import necessary libraries
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer, setup_chat_format
import torch

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)

# Load the model and tokenizer
model_name = "HuggingFaceTB/SmolLM2-135M"

model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=model_name
).to(device)
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_name)

# Set up the chat format
model, tokenizer = setup_chat_format(model=model, tokenizer=tokenizer)

# Set our name for the finetune to be saved &/ uploaded to
finetune_name = "SmolLM2-FT-MyDataset"
finetune_tags = ["smol-course", "module_1"]

### Example output

In [20]:
from random import randint
eval_dataset = load_dataset(
    "json", data_files="sft_data/test_*.json", split="train")
rand_idx = randint(0, len(eval_dataset))
prompt = tokenizer.apply_chat_template(eval_dataset[rand_idx]["messages"][:2], tokenize=False, add_generation_prompt=True)
print(prompt)
inputs = tokenizer(prompt, return_tensors="pt").to(device)
outputs = model.generate(**inputs, max_new_tokens=100)
print("Before training:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Generating train split: 0 examples [00:00, ? examples/s]

<|im_start|>system
You are an AI assistant tasked with extracting ingredients from a recipe text. Here's what you need to do:

1. Read the following recipe text carefully.

2. Identify all ingredients mentioned in the text. Look for:
   - Specific food items
   - Quantities (e.g., numbers, fractions, or words like "cup", "tablespoon", "pinch")
   - Any descriptors of the ingredients (e.g., "chopped", "diced", "fresh")

3. Create a list of all identified ingredients. Each ingredient should be a complete phrase including its quantity and any descriptors.

4. Output your list of ingredients as a Python list of strings. Each ingredient should be a separate element in the list.

For example:
['3 shallots', '6 tablespoons (3/4 stick) butter, room temperature', '4 teaspoons minced fresh rosemary', '1 tablespoon grated orange peel']

Remember to include all ingredients, even if they seem minor or are used in small quantities.
<|im_end|>
<|im_start|>user
Extract all ingredients from this recipe

The `SFTTrainer`  supports a native integration with `peft`, which makes it super easy to efficiently tune LLMs using, e.g. LoRA. We only need to create our `LoraConfig` and provide it to the trainer.

<div style='background-color: lightblue; padding: 10px; border-radius: 5px; margin-bottom: 20px; color:black'>
    <h2 style='margin: 0;color:blue'>Exercise: Define LoRA parameters for finetuning</h2>
    <p>Take a dataset from the Hugging Face hub and finetune a model on it. </p>
    <p><b>Difficulty Levels</b></p>
    <p>🐢 Use the general parameters for an abitrary finetune</p>
    <p>🐕 Adjust the parameters and review in weights & biases.</p>
    <p>🦁 Adjust the parameters and show change in inference results.</p>
</div>

In [21]:
from peft import LoraConfig

# TODO: Configure LoRA parameters
# r: rank dimension for LoRA update matrices (smaller = more compression)
rank_dimension = 6
# lora_alpha: scaling factor for LoRA layers (higher = stronger adaptation)
lora_alpha = 8
# lora_dropout: dropout probability for LoRA layers (helps prevent overfitting)
lora_dropout = 0.05

peft_config = LoraConfig(
    r=rank_dimension,  # Rank dimension - typically between 4-32
    lora_alpha=lora_alpha,  # LoRA scaling factor - typically 2x rank
    lora_dropout=lora_dropout,  # Dropout probability for LoRA layers
    bias="none",  # Bias type for LoRA. the corresponding biases will be updated during training.
    target_modules="all-linear",  # Which modules to apply LoRA to
    task_type="CAUSAL_LM",  # Task type for model architecture
)

Before we can start our training we need to define the hyperparameters (`TrainingArguments`) we want to use.

In [22]:
# Training configuration
# Hyperparameters based on QLoRA paper recommendations
args = SFTConfig(
    # Output settings
    output_dir=finetune_name,  # Directory to save model checkpoints
    # Training duration
    num_train_epochs=3,  # Number of training epochs
    # Batch size settings
    per_device_train_batch_size=2,  # Batch size per GPU
    gradient_accumulation_steps=2,  # Accumulate gradients for larger effective batch
    # Memory optimization
    gradient_checkpointing=True,  # Trade compute for memory savings
    # Optimizer settings
    optim="adamw_torch_fused",  # Use fused AdamW for efficiency
    learning_rate=2e-4,  # Learning rate (QLoRA paper)
    max_grad_norm=0.3,  # Gradient clipping threshold
    # Learning rate schedule
    warmup_ratio=0.03,  # Portion of steps for warmup
    lr_scheduler_type="constant",  # Keep learning rate constant after warmup
    # Logging and saving
    logging_steps=10,  # Log metrics every N steps
    save_strategy="epoch",  # Save checkpoint every epoch
    # Precision settings
    bf16=True,  # Use bfloat16 precision
    # Integration settings
    push_to_hub=False,  # Don't push to HuggingFace Hub
    report_to=None,  # Disable external logging
)

We now have every building block we need to create our `SFTTrainer` to start then training our model.

In [23]:
max_seq_length = 3072#1512  # max sequence length for model and packing of the dataset

# Create SFTTrainer with LoRA configuration
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    peft_config=peft_config,  # LoRA configuration
    #max_seq_length=max_seq_length,  # Maximum sequence length
    tokenizer=tokenizer,
    #packing=True,  # Enable input packing for efficiency
    #dataset_kwargs={
     #   "add_special_tokens": False,  # Special tokens handled by template
      #  "append_concat_token": False,  # No additional separator needed
    #},
)

<ipython-input-23-1218b5d12753>:4: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  trainer = SFTTrainer(


Map:   0%|          | 0/12150 [00:00<?, ? examples/s]

Start training our model by calling the `train()` method on our `Trainer` instance. This will start the training loop and train our model for 3 epochs. Since we are using a PEFT method, we will only save the adapted model weights and not the full model.

In [24]:
# start training, the model will be automatically saved to the hub and the output directory
trainer.train()

# save model
trainer.save_model()
if os.getenv("HF_TOKEN"):
    trainer.push_to_hub(tags=finetune_tags)

<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

 ··········


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
10,2.334800
20,2.208400
30,2.080800
40,1.845500
50,1.677600
60,1.551900
70,1.451400
80,1.311800
90,1.288000
100,1.337200


The training with Flash Attention for 3 epochs with a dataset of 15k samples took 4:14:36 on a `g5.2xlarge`. The instance costs `1.21$/h` which brings us to a total cost of only ~`5.3$`.



### Merge LoRA Adapter into the Original Model

When using LoRA, we only train adapter weights while keeping the base model frozen. During training, we save only these lightweight adapter weights (~2-10MB) rather than a full model copy. However, for deployment, you might want to merge the adapters back into the base model for:

1. **Simplified Deployment**: Single model file instead of base model + adapters
2. **Inference Speed**: No adapter computation overhead
3. **Framework Compatibility**: Better compatibility with serving frameworks


In [25]:
from peft import AutoPeftModelForCausalLM


# Load PEFT model on CPU
model = AutoPeftModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=args.output_dir,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

# Merge LoRA and base model and save
merged_model = model.merge_and_unload()
merged_model.save_pretrained(
    args.output_dir, safe_serialization=True, max_shard_size="2GB"
)

## 3. Test Model and run Inference

After the training is done we want to test our model. We will load different samples from the original dataset and evaluate the model on those samples, using a simple loop and accuracy as our metric.



<div style='background-color: lightblue; padding: 10px; border-radius: 5px; margin-bottom: 20px; color:black'>
    <h2 style='margin: 0;color:blue'>Bonus Exercise: Load LoRA Adapter</h2>
    <p>Use what you learnt from the ecample note book to load your trained LoRA adapter for inference.</p>
</div>

In [26]:
# free the memory again
del model
del trainer
torch.cuda.empty_cache()

In [27]:
import torch
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer, pipeline

# Load Model with PEFT adapter
tokenizer = AutoTokenizer.from_pretrained(finetune_name)
model = AutoPeftModelForCausalLM.from_pretrained(
    finetune_name, device_map="auto", torch_dtype=torch.float16
)
pipe = pipeline(
    "text-generation", model=merged_model, tokenizer=tokenizer, device=device
)

Device set to use cuda


In [28]:
prompts = [
    "What is the capital of Germany? Explain why thats the case and if it was different in the past?",
    "Write a Python function to calculate the factorial of a number.",
    "A rectangular garden has a length of 25 feet and a width of 15 feet. If you want to build a fence around the entire garden, how many feet of fencing will you need?",
    "What is the difference between a fruit and a vegetable? Give examples of each.",
]


def test_inference(prompt):
    prompt = pipe.tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
    outputs = pipe(
        prompt,
    )
    return outputs[0]["generated_text"][len(prompt) :].strip()


for prompt in prompts:
    print(f"    prompt:\n{prompt}")
    print(f"    response:\n{test_inference(prompt)}")
    print("-" * 50)

    prompt:
What is the capital of Germany? Explain why thats the case and if it was different in the past?
    response:
What is the capital of Germany? Explain why thats the case and if it was different in the
--------------------------------------------------
    prompt:
Write a Python function to calculate the factorial of a number.
    response:
["1", "2", "3", "4", "5", "6", "7
--------------------------------------------------
    prompt:
A rectangular garden has a length of 25 feet and a width of 15 feet. If you want to build a fence around the entire garden, how many feet of fencing will you need?
    response:
["15-foot fence", "15-foot fence", "15-foot fence
--------------------------------------------------
    prompt:
What is the difference between a fruit and a vegetable? Give examples of each.
    response:
What is the difference between a fruit and a vegetable? Give examples of each.
A fruit is
--------------------------------------------------


Lets test some prompt samples and see how the model performs.

Conlcusion, the model starts to adapt to the task, however likely there is not enough information to accurately extract the ingredient sizes from the description alone. A pre-processing step is to remove all quantifiers.

In [32]:
prompts = [
    "What is the capital of Germany? Explain why thats the case and if it was different in the past?",
    "Write a Python function to calculate the factorial of a number.",
    "A rectangular garden has a length of 25 feet and a width of 15 feet. If you want to build a fence around the entire garden, how many feet of fencing will you need?",
    "What is the difference between a fruit and a vegetable? Give examples of each.",
]


def test_inference(prompt):
    prompt = pipe.tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
    outputs = pipe(
        prompt,
    )
    return outputs[0]["generated_text"][len(prompt) :].strip()

eval_dataset = load_dataset(
    "json", data_files="sft_data/test_*.json", split="train")
from random import randint
for i in range(5):
    rand_idx = randint(0, len(eval_dataset))
    prompt = tokenizer.apply_chat_template(eval_dataset[rand_idx]["messages"][:2], tokenize=False, add_generation_prompt=True)

    print(f"    prompt:\n{prompt}")
    print(f"    response:\n{test_inference(prompt)}")
    print("-" * 50)

    prompt:
<|im_start|>system
You are an AI assistant tasked with extracting ingredients from a recipe text. Here's what you need to do:

1. Read the following recipe text carefully.

2. Identify all ingredients mentioned in the text. Look for:
   - Specific food items
   - Quantities (e.g., numbers, fractions, or words like "cup", "tablespoon", "pinch")
   - Any descriptors of the ingredients (e.g., "chopped", "diced", "fresh")

3. Create a list of all identified ingredients. Each ingredient should be a complete phrase including its quantity and any descriptors.

4. Output your list of ingredients as a Python list of strings. Each ingredient should be a separate element in the list.

For example:
['3 shallots', '6 tablespoons (3/4 stick) butter, room temperature', '4 teaspoons minced fresh rosemary', '1 tablespoon grated orange peel']

Remember to include all ingredients, even if they seem minor or are used in small quantities.
<|im_end|>
<|im_start|>user
Extract all ingredients from